# PGx Analysis - Cohort Runner

This notebook runs **PGx Analysis** for all configured cohorts and age bands.

- **Script**: `5c_pgx_analysis/run_analysis.py`
- **Purpose**: Map drugs to genes and create pharmacogenomic features
- **Outputs**: PGx features added to model data

## Features

✅ **Dynamic cohort selection** - Configure which cohorts/age bands to run  
✅ **Idempotent** - Automatically skips completed analyses  
✅ **S3 integration** - Results automatically synced to S3

## Usage

1. Configure cohorts and age bands in the configuration cell below
2. Run the execution cell to process all combinations
3. Results are saved locally and synced to S3 automatically

In [ ]:
# Environment Setup

import os
import sys
from pathlib import Path

def resolve_python_bin() -> Path:
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)

PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Using Python binary: {PYTHON_BIN}")

def resolve_project_root() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parents[1]
    notebook_path = Path(os.getcwd()).resolve()
    if notebook_path.name == "5c_pgx_analysis":
        return notebook_path.parent
    for parent in notebook_path.parents:
        if parent.name == "pgx-analysis":
            return parent
    return notebook_path

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from py_helpers.constants import COHORT_NAMES, AGE_BANDS

## Configuration

Select which cohorts and age bands to process. Leave empty lists to process all.

In [ ]:
# Configuration: Select cohorts and age bands to process
COHORTS_TO_RUN = []  # e.g., ['opioid_ed'] or [] for all
AGE_BANDS_TO_RUN = []  # e.g., ['0-12', '13-24'] or [] for all

if not COHORTS_TO_RUN:
    COHORTS_TO_RUN = COHORT_NAMES.copy()
if not AGE_BANDS_TO_RUN:
    AGE_BANDS_TO_RUN = AGE_BANDS.copy()

print(f"Will process {len(COHORTS_TO_RUN)} cohort(s) and {len(AGE_BANDS_TO_RUN)} age band(s)")
print(f"Cohorts: {COHORTS_TO_RUN}")
print(f"Age bands: {AGE_BANDS_TO_RUN}")

## Run PGx Analysis

In [ ]:
# Run PGx Analysis for all configured cohort/age band combinations

import subprocess

FAIL_FAST = True  # Stop on first failure; set to False to continue on errors

combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Running PGx Analysis analysis for {len(combinations)} combinations...")
print("=" * 80)

for cohort_name, age_band in combinations:
    print(f"\n[PGx Analysis] Starting: {cohort_name} / {age_band}")
    print("-" * 80)
    
    result = subprocess.run(
        [str(PYTHON_BIN), "5c_pgx_analysis/run_analysis.py",
         "--cohort-name", cohort_name,
         "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    
    if result.returncode == 0:
        print(f"[PGx Analysis] COMPLETED: {cohort_name} / {age_band}")
    else:
        msg = f"[PGx Analysis] FAILED ({result.returncode}): {cohort_name} / {age_band}"
        print(msg)
        if FAIL_FAST:
            raise RuntimeError(msg)

print("\n" + "=" * 80)
print("All PGx Analysis analyses completed.")
print("=" * 80)